# 🏙️ Airbnb Vancouver — Market Segmentation & Sentiment Analysis

**Author:** Gowri  
**Dataset:** Inside Airbnb — Vancouver, BC  
**Records:** ~4,705 listings · ~277,819 reviews  
**Tools:** Python · Pandas · Scikit-learn · VADER · Matplotlib · Seaborn  

---

## 📌 Project Objective

This project analyses the Vancouver Airbnb market to:

1. **Understand** the distribution of prices, room types, and neighbourhoods through exploratory data analysis (EDA)
2. **Segment** listings into meaningful market clusters using K-Means + PCA
3. **Decode** guest sentiment by applying VADER NLP sentiment analysis to 277K+ reviews
4. **Combine** clustering and sentiment insights to identify what drives guest satisfaction per segment

---

## 📂 Dataset

| File | Description |
|------|-------------|
| `listings.csv` | One row per listing — price, room type, neighbourhood, availability, host info |
| `reviews.csv` | One row per review — listing ID, reviewer, date, and free-text comment |

> **Source:** [Inside Airbnb](http://insideairbnb.com/get-the-data/) — publicly available Airbnb data scraped for Vancouver, BC.

---

## 🗂️ Notebook Structure

| Section | Description |
|---------|-------------|
| [1. Setup & Data Loading](#1-setup--data-loading) | Import libraries, load CSVs, preview raw data |
| [2. Data Cleaning & Feature Selection](#2-data-cleaning--feature-selection) | Select relevant columns, fix dtypes, handle missing values |
| [3. Exploratory Data Analysis (EDA)](#3-exploratory-data-analysis-eda) | Price, room type, neighbourhood, reviews, correlation |
| [4. Outlier Detection & Treatment](#4-outlier-detection--treatment) | IQR-based detection, capping strategy |
| [5. Market Segmentation — K-Means Clustering](#5-market-segmentation--k-means-clustering) | Elbow method, K=4 clusters, PCA visualisation |
| [6. Sentiment Analysis — VADER NLP](#6-sentiment-analysis--vader-nlp) | Text cleaning, compound scoring, distribution |
| [7. Cluster × Sentiment Deep Dive](#7-cluster--sentiment-deep-dive) | Sentiment breakdown per cluster, word clouds |
| [8. Key Takeaways](#8-key-takeaways) | Business insights summary |


## 1. Setup & Data Loading

### 1.1 Import Libraries

Core libraries used throughout the notebook:
- **pandas / numpy** — data manipulation
- **matplotlib / seaborn** — visualisation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Consistent plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100


### 1.2 Load Raw Data

In [ ]:
# Load the two core datasets from Inside Airbnb
df_listings = pd.read_csv("listings.csv")
df_reviews  = pd.read_csv("reviews.csv")


### 1.3 Initial Preview

In [ ]:
print("=== Listings Dataset ===")
print(f"Shape: {df_listings.shape}")
df_listings.head(3)


In [ ]:
print("=== Reviews Dataset ===")
print(f"Shape: {df_reviews.shape}")
df_reviews.head(3)


In [ ]:
# Structural overview — dtypes, non-null counts
print("--- Listings Info ---")
df_listings.info()
print("\n--- Reviews Info ---")
df_reviews.info()


In [ ]:
# Numeric summary statistics
print("--- Listings Describe ---")
display(df_listings.describe())
print("\n--- Reviews Describe ---")
display(df_reviews.describe())


## 2. Data Cleaning & Feature Selection

### 2.1 Select Relevant Columns

The raw listings CSV contains 70+ columns. We retain only those relevant to EDA, 
clustering, and BI reporting — reducing noise and memory footprint.


In [ ]:
columns_to_keep = [
    "id",
    "host_id",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "latitude",
    "longitude",
    "room_type",
    "price",
    "minimum_nights",
    "number_of_reviews",
    "last_review",
    "reviews_per_month",
    "calculated_host_listings_count",
    "availability_365",
    "number_of_reviews_ltm",
    "license"
]

# Subset and copy to avoid SettingWithCopyWarning
df_listings_n = df_listings[columns_to_keep].copy()

print(f"Reduced listings shape: {df_listings_n.shape}")
df_listings_n.head()


### 2.2 Check Missing Values

In [ ]:
missing = df_listings_n.isnull().sum()
missing_pct = (missing / len(df_listings_n) * 100).round(2)

missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)


> **Note:** `last_review` and `reviews_per_month` are naturally missing for 
> listings with zero reviews — these will be handled contextually in later sections.


### 2.3 Fix Price Column (String → Float)

In [ ]:
# Price arrives as a string with '$' and ',' characters (e.g. '$1,250.00')
# Strip formatting and convert to float for numerical analysis
df_listings_n["price"] = (
    df_listings_n["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

print("Price dtype after conversion:", df_listings_n["price"].dtype)
print(df_listings_n["price"].describe())


## 3. Exploratory Data Analysis (EDA)

### 3.1 Price Distribution

We cap the visualisation at $1,000 to exclude extreme outliers and 
reveal the shape of the mainstream market.


In [ ]:
df_price_filtered = df_listings_n[df_listings_n["price"] < 1000]

plt.figure(figsize=(9, 5))
sns.histplot(df_price_filtered["price"], bins=50, kde=True, color="steelblue")
plt.title("Price Distribution (Listings < $1,000/night)", fontsize=14)
plt.xlabel("Price (USD)")
plt.ylabel("Number of Listings")
plt.tight_layout()
plt.show()


**💡 Insight — Price Distribution**

- The distribution is **right-skewed**: most listings cluster between **$100–$250/night**.
- Prices above $500 are rare outliers, typically luxury or large multi-bedroom properties.
- The bulk of hosts compete in the **budget-to-midrange market** ($50–$300).


### 3.2 Room Type Distribution

In [ ]:
plt.figure(figsize=(7, 4))
df_listings_n["room_type"].value_counts().plot(
    kind="bar", color=["steelblue", "seagreen", "salmon", "gold"]
)
plt.title("Room Type Distribution", fontsize=14)
plt.xlabel("Room Type")
plt.ylabel("Number of Listings")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


**💡 Insight — Room Type**

- **Entire home/apt** dominates — travellers strongly prefer privacy and full-unit stays.
- **Private rooms** form a significant secondary segment, attracting budget and solo travellers.
- **Shared rooms** are extremely rare, reflecting low demand for hostel-style accommodation.
- Room type is a **primary driver** of price, occupancy, and cluster assignment.


### 3.3 Top 10 Neighbourhoods by Listing Count

In [ ]:
plt.figure(figsize=(12, 5))
df_listings_n["neighbourhood"].value_counts().head(10).plot(
    kind="bar", color="steelblue"
)
plt.title("Top 10 Neighbourhoods by Number of Listings", fontsize=14)
plt.xlabel("Neighbourhood")
plt.ylabel("Number of Listings")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


**💡 Insight — Neighbourhood Supply**

- **Downtown** has the highest listing concentration by a significant margin.
- Tourist-friendly areas like **Kitsilano** and **Mount Pleasant** show strong Airbnb activity.
- Residential neighbourhoods have moderate supply — potentially less competitive for new hosts.


### 3.4 Reviews Per Month Distribution

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(
    df_listings_n["reviews_per_month"].dropna(),
    bins=30, kde=True, color="mediumseagreen"
)
plt.title("Reviews Per Month Distribution", fontsize=14)
plt.xlabel("Reviews per Month")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


**💡 Insight — Review Frequency**

- Most listings receive **fewer than 2 reviews per month**, indicating moderate-to-low occupancy.
- A small cohort of high-performing listings accumulates reviews rapidly — likely Superhost properties.
- Missing values in this column map to listings that have **never been reviewed**.


### 3.5 Price by Room Type (Box Plot)

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df_price_filtered,
    x="room_type",
    y="price",
    palette=["steelblue", "seagreen", "salmon", "gold"]
)
plt.title("Price Distribution by Room Type", fontsize=14)
plt.xlabel("Room Type")
plt.ylabel("Price (USD)")
plt.tight_layout()
plt.show()


**💡 Insight — Price vs Room Type**

- **Entire homes** command the highest median prices and widest price range.
- **Private rooms** are mid-priced with tighter variation.
- **Shared rooms** are cheapest with minimal pricing flexibility.
- Room type is one of the most predictive features for pricing models.


### 3.6 Top 10 Neighbourhoods by Median Price

In [ ]:
neigh_price = (
    df_price_filtered
    .groupby("neighbourhood")["price"]
    .median()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
neigh_price.plot(kind="bar", color="steelblue")
plt.title("Top 10 Neighbourhoods by Median Listing Price", fontsize=14)
plt.xlabel("Neighbourhood")
plt.ylabel("Median Price (USD)")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.show()


**💡 Insight — Neighbourhood Pricing**

- **Downtown** leads on median price, reflecting premium demand near tourist attractions.
- **Kitsilano** and **Downtown Eastside** also command above-average rates.
- Mid-range neighbourhoods (South Cambie, West End, Strathcona) cluster around $160–$180/night.
- Price variation is heavily influenced by **proximity to downtown and transit**.


### 3.7 Correlation Heatmap

In [ ]:
# Select numeric columns only; drop IDs and coordinates (not meaningful for correlation)
numeric_cols = df_listings_n.select_dtypes(include=["int64", "float64"]).drop(
    columns=["id", "host_id", "latitude", "longitude"], errors="ignore"
)

plt.figure(figsize=(12, 8))
sns.heatmap(
    numeric_cols.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)
plt.title("Correlation Heatmap — Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()


**💡 Insight — Feature Correlations**

- `price` has **weak correlations** with all other numeric features — pricing is complex and context-dependent.
- Review count and review frequency are moderately correlated, as expected.
- `minimum_nights` has a mild positive correlation with `availability_365` (0.39) — longer-stay listings tend to stay available longer.
- Host listing count does not strongly influence pricing or review scores.


## 4. Outlier Detection & Treatment

### 4.1 IQR-Based Outlier Detection

We use the **Interquartile Range (IQR)** method to flag outliers across all numeric columns.  
A value is considered an outlier if it falls below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR`.


In [ ]:
def detect_outliers_all(df):
    """Return a DataFrame counting IQR-based outliers per numeric column."""
    outlier_summary = {}
    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        count = df[(df[col] < lower) | (df[col] > upper)][col].count()
        outlier_summary[col] = count

    return pd.DataFrame.from_dict(
        outlier_summary, orient="index", columns=["Outlier_Count"]
    ).sort_values("Outlier_Count", ascending=False)

outlier_report = detect_outliers_all(df_listings_n)
outlier_report


### 4.2 Outlier Treatment Strategy

In [ ]:
# Strategy: CAP rather than DROP to preserve as many listings as possible
# 1. Cap price at 99th percentile (removes extreme luxury outliers)
upper_price = df_listings_n["price"].quantile(0.99)
df_listings_n["price"] = np.where(
    df_listings_n["price"] > upper_price,
    upper_price,
    df_listings_n["price"]
)

# 2. Cap minimum_nights at 30 days — beyond this, it's a long-term rental, not Airbnb
df_listings_n["minimum_nights"] = df_listings_n["minimum_nights"].clip(upper=30)

print(f"Price capped at 99th percentile: ${upper_price:.2f}")
print(f"minimum_nights capped at 30 nights")
print("Outlier treatment complete ✓")


## 5. Market Segmentation — K-Means Clustering

We segment Vancouver Airbnb listings into distinct market groups using **K-Means clustering**.

**Pipeline:**
1. Feature selection → numeric + encoded categorical features
2. Impute missing values (median for numeric, mode for categorical)
3. Label-encode `room_type` and `neighbourhood`
4. Standardise all features with `StandardScaler`
5. Determine optimal K via the **Elbow Method**
6. Fit K-Means with K=4 and assign cluster labels
7. Visualise with **PCA** (2D projection)


### 5.1 Prepare Clustering Features

In [ ]:
# Select features relevant to listing behaviour and market positioning
df_cluster = df_listings_n[[
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "availability_365",
    "calculated_host_listings_count",
    "room_type",
    "neighbourhood"
]].copy()

# Impute numeric columns with column median
num_cols = [
    "price", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "availability_365", "calculated_host_listings_count"
]
for col in num_cols:
    df_cluster[col] = df_cluster[col].fillna(df_cluster[col].median())

# Impute categoricals with mode
cat_cols = ["room_type", "neighbourhood"]
for col in cat_cols:
    df_cluster[col] = df_cluster[col].fillna(df_cluster[col].mode()[0])

print(f"Clustering dataset shape: {df_cluster.shape}")
print(f"Missing values remaining: {df_cluster.isnull().sum().sum()}")


### 5.2 Encode & Scale

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Label-encode categorical features
le = LabelEncoder()
for col in cat_cols:
    df_cluster[col] = le.fit_transform(df_cluster[col])

# Standardise all features — essential for distance-based algorithms
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

print("Feature scaling complete. Shape:", X_scaled.shape)


### 5.3 Elbow Method — Find Optimal K

In [ ]:
from sklearn.cluster import KMeans

inertia = []

for k in range(2, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), inertia, marker="o", color="steelblue", linewidth=2)
plt.title("Elbow Method — Optimal Number of Clusters", fontsize=14)
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.xticks(range(2, 11))
plt.tight_layout()
plt.show()


**📌 Elbow Decision:** The most significant drop in inertia occurs between K=3 and K=4.  
Gains diminish from K=4 onward, making **K=4** the optimal choice.


### 5.4 Fit K-Means (K=4)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df_cluster["cluster"] = kmeans.fit_predict(X_scaled)
df_listings_n["cluster"] = df_cluster["cluster"]

print("K-Means clustering complete ✓")
print("Cluster distribution:")
print(df_listings_n["cluster"].value_counts().sort_index())


### 5.5 Cluster Profile Summary

In [ ]:
cluster_summary = df_listings_n.groupby("cluster").agg(
    Avg_Price       = ("price",                         "mean"),
    Avg_Min_Nights  = ("minimum_nights",                "mean"),
    Avg_Reviews     = ("number_of_reviews",             "mean"),
    Avg_Avail_365   = ("availability_365",              "mean"),
    Avg_Host_Listings = ("calculated_host_listings_count", "mean"),
    Count           = ("cluster",                       "count")
).round(2)

cluster_summary


### 5.6 Cluster Interpretation

| Cluster | Name | Characteristics |
|---------|------|-----------------|
| **0** | 🟣 Budget Small Listings | Cheap, small, long min. stay, lower review volume |
| **1** | 🔵 High-Performing Mid-Range | Flexible, affordable, many reviews, high occupancy |
| **2** | 🟢 Premium Large Family Homes | Expensive, spacious, suited for group stays |
| **3** | 🟡 Long-Term Professional Host Units | 30-night minimum, multi-listing professional hosts |


### 5.7 Cluster Size Distribution

In [ ]:
plt.figure(figsize=(8, 5))
df_listings_n["cluster"].value_counts().sort_index().plot(
    kind="bar",
    color=["mediumpurple", "steelblue", "seagreen", "gold"]
)
plt.title("Cluster Sizes — Listings per Segment", fontsize=14)
plt.xlabel("Cluster")
plt.ylabel("Number of Listings")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 5.8 PCA Visualisation of Clusters

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2 principal components for 2D visualisation
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame({
    "PCA1":    pca_result[:, 0],
    "PCA2":    pca_result[:, 1],
    "cluster": df_listings_n["cluster"]
})

explained = pca.explained_variance_ratio_
print(f"Variance explained — PCA1: {explained[0]:.1%}, PCA2: {explained[1]:.1%}")
print(f"Total variance captured: {sum(explained):.1%}")


In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(
    data=df_pca,
    x="PCA1", y="PCA2",
    hue="cluster",
    palette="viridis",
    alpha=0.5,
    s=20
)
plt.title("PCA Projection of K-Means Clusters (K=4)", fontsize=14)
plt.xlabel(f"PCA Component 1 ({explained[0]:.1%} variance)")
plt.ylabel(f"PCA Component 2 ({explained[1]:.1%} variance)")
plt.legend(title="Cluster", labels=["0: Budget", "1: Mid-Range", "2: Premium", "3: Long-Term"])
plt.tight_layout()
plt.show()


**💡 PCA Cluster Insights**

- **Four visually distinct groups** confirm K-Means produced meaningful segments.
- **Cluster 1 (Mid-Range)** is the largest and spreads across a wide region — natural market diversity.
- **Cluster 0 (Budget)** sits tightly on the left — very consistent, homogeneous features.
- **Cluster 2 (Premium)** extends far right on PCA1 — driven by price and size features.
- **Cluster 3 (Long-Term)** rises above others on PCA2 — driven by minimum_nights and host listing count.


## 6. Sentiment Analysis — VADER NLP

We apply **VADER (Valence Aware Dictionary and sEntiment Reasoner)** to 277,819 guest reviews.

VADER is particularly well-suited for short, informal social media/review text — it handles 
emoji, capitalisation, and punctuation as sentiment signals without requiring model training.

**Pipeline:**
1. Merge reviews with listings (left join on `listing_id`)
2. Drop rows with null comments
3. Clean text (lowercase, strip HTML, URLs, punctuation)
4. Score each review with VADER compound score (−1 to +1)
5. Label: positive (≥ 0.05), negative (≤ −0.05), neutral (otherwise)


### 6.1 Merge Listings & Reviews

In [ ]:
# Rename review 'id' to avoid column clash after merge
df_reviews = df_reviews.rename(columns={"id": "review_id"})

# Left join: every review gets its listing's metadata
df_merged = df_reviews.merge(
    df_listings,
    left_on="listing_id",
    right_on="id",
    how="left"
)

# Drop reviews without comment text
df_merged = df_merged.dropna(subset=["comments"])

print(f"Merged dataset shape: {df_merged.shape}")
print(f"Reviews with comments: {len(df_merged):,}")


### 6.2 Text Cleaning

In [ ]:
import re
import string

def clean_text(text):
    """
    Normalise raw review text for NLP processing:
    - Lowercase
    - Remove HTML tags
    - Remove URLs
    - Strip punctuation
    - Collapse whitespace
    """
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)           # remove HTML tags
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )                                            # strip punctuation
    text = re.sub(r"\s+", " ", text).strip()     # collapse whitespace
    return text

df_merged["cleaned_comments"] = df_merged["comments"].apply(clean_text)

print("Text cleaning complete ✓")
df_merged[["comments", "cleaned_comments"]].head(3)


### 6.3 VADER Sentiment Scoring

In [ ]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download("vader_lexicon", quiet=True)
sid = SentimentIntensityAnalyzer()

def get_sentiment(text):
    """
    Classify review sentiment using VADER compound score:
      >= +0.05  → Positive
      <= −0.05  → Negative
      Otherwise → Neutral
    """
    score = sid.polarity_scores(text)["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df_merged["sentiment"] = df_merged["cleaned_comments"].apply(get_sentiment)

print("Sentiment scoring complete ✓")
print(df_merged["sentiment"].value_counts())


### 6.4 Overall Sentiment Distribution

In [ ]:
sentiment_counts = df_merged["sentiment"].value_counts()

plt.figure(figsize=(7, 4))
sentiment_counts.plot(
    kind="bar",
    color=["seagreen", "lightgrey", "salmon"]
)
plt.title("Sentiment Distribution — All Airbnb Reviews", fontsize=14)
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**💡 Insight — Overall Sentiment**

- **Positive reviews dominate (~80–90%)**, reflecting high guest satisfaction across Vancouver listings.
- **Neutral reviews (~5–8%)** represent average or mixed experiences.
- **Negative reviews (<5%)** are rare — likely related to cleanliness, communication, or unmet expectations.
- Overall, Vancouver Airbnb hosts maintain strong guest experience quality.


### 6.5 Word Clouds — Positive vs Negative Reviews

In [ ]:
from wordcloud import WordCloud

positive_text = " ".join(df_merged[df_merged["sentiment"] == "positive"]["cleaned_comments"])
negative_text = " ".join(df_merged[df_merged["sentiment"] == "negative"]["cleaned_comments"])

# Positive Word Cloud
wc_positive = WordCloud(
    width=1200, height=500,
    background_color="white",
    colormap="Greens",
    max_words=200
).generate(positive_text)

plt.figure(figsize=(14, 6))
plt.imshow(wc_positive, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud — Positive Reviews", fontsize=14)
plt.tight_layout()
plt.show()

# Negative Word Cloud
wc_negative = WordCloud(
    width=1200, height=500,
    background_color="white",
    colormap="Reds",
    max_words=200
).generate(negative_text)

plt.figure(figsize=(14, 6))
plt.imshow(wc_negative, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud — Negative Reviews", fontsize=14)
plt.tight_layout()
plt.show()


## 7. Cluster × Sentiment Deep Dive

We now combine our **K-Means segment labels** with **VADER sentiment scores** 
to answer: *Which market segment generates the most positive guest experiences?*


### 7.1 Attach Cluster Labels to Reviews

In [ ]:
# Bring cluster assignments into the merged reviews DataFrame
df_merged = df_merged.merge(
    df_listings_n[["id", "cluster"]],
    how="left",
    left_on="listing_id",
    right_on="id"
)

print(f"Reviews with cluster assignment: {df_merged['cluster'].notna().sum():,}")


### 7.2 Sentiment × Cluster Breakdown

In [ ]:
# Count of each sentiment per cluster
sentiment_cluster = (
    df_merged
    .groupby(["cluster", "sentiment"])
    .size()
    .unstack()
    .fillna(0)
)

# Percentage within each cluster
sentiment_cluster_pct = (
    sentiment_cluster.div(sentiment_cluster.sum(axis=1), axis=0) * 100
).round(2)

print("Sentiment distribution (%) per cluster:")
sentiment_cluster_pct


In [ ]:
cluster_labels = {
    0: "0: Budget",
    1: "1: Mid-Range",
    2: "2: Premium",
    3: "3: Long-Term"
}

ax = sentiment_cluster_pct.rename(index=cluster_labels).plot(
    kind="bar",
    figsize=(10, 6),
    color=["seagreen", "lightgrey", "salmon"]
)
plt.title("Sentiment Distribution (%) by Market Segment", fontsize=14)
plt.ylabel("Percentage (%)")
plt.xlabel("Cluster")
plt.xticks(rotation=20, ha="right")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()


### 7.3 Per-Cluster Word Clouds

In [ ]:
cluster_config = [
    (0, "Blues",   "0 — Budget Small Listings"),
    (1, "Greens",  "1 — High-Performing Mid-Range"),
    (2, "Oranges", "2 — Premium Large Family Homes"),
    (3, "Reds",    "3 — Long-Term Professional Host Units"),
]

for cluster_id, colormap, title in cluster_config:
    cluster_text = " ".join(
        df_merged[df_merged["cluster"] == cluster_id]["cleaned_comments"]
    )
    if not cluster_text.strip():
        print(f"Cluster {cluster_id}: no review text found — skipping.")
        continue

    wc = WordCloud(
        width=1200, height=500,
        background_color="white",
        colormap=colormap,
        max_words=200
    ).generate(cluster_text)

    plt.figure(figsize=(14, 6))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word Cloud — Cluster {title}", fontsize=14)
    plt.tight_layout()
    plt.show()


## 8. Key Takeaways

### 📊 Market Landscape
- Vancouver's Airbnb market is **dominated by entire homes** priced primarily between $100–$250/night.
- **Downtown** commands both the highest listing supply and highest median prices.
- Price has **weak linear correlations** with other numeric features — location, room type, and host quality matter more.

### 🔍 Market Segments (K=4 Clusters)
| Segment | Strategy Implication |
|---------|----------------------|
| 🟣 Budget (Cluster 0) | Price-sensitive travellers; hosts should focus on cleanliness and communication |
| 🔵 Mid-Range (Cluster 1) | Highest volume, most competitive — differentiate via reviews and flexibility |
| 🟢 Premium (Cluster 2) | Group and family stays; justify price with amenities and space |
| 🟡 Long-Term (Cluster 3) | Corporate/relocation market; optimise for monthly booking channels |

### 💬 Guest Sentiment
- **>80% of reviews are positive**, confirming generally high satisfaction.
- Negative reviews are rare but concentrated around **cleanliness, noise, and host responsiveness**.
- **Premium listings (Cluster 2)** tend to attract the most detailed reviews — guests have higher expectations.

### 🔗 Next Steps
- Export cluster labels to Power BI / Tableau for geographic and trend dashboards
- Build a **price prediction model** using cluster membership as a feature
- Track **sentiment trends over time** using the `date` field in reviews
